# 05c — VulBERTa Cache Builder

Builds a VulBERTa token cache that matches Anas's GCB cache exactly.

| | Anas GCB cache | VulBERTa cache (this notebook) |
|---|---|---|
| Records | 14,522 chunks | 14,522 chunks |
| Functions | 4,085 | 4,085 |
| Max length | 512 | 512 |
| Tokenizer | GraphCodeBERT | VulBERTa-MLP-D2A |
| Order | reference | must match exactly |

**Key principle:** record `i` in VulBERTa cache = same code chunk as record `i` in Anas's cache.
We reconstruct each chunk from UNIFIED.jsonl using Anas's `global_ids` and `chunk_index`.

## 0 — Install

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'torch', 'pyyaml', 'tqdm',
], check=False)
print('Done.')


## 1 — Paths & Config

In [ ]:
import os, sys, json, platform
from pathlib import Path
from collections import Counter
import yaml
import torch

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/dataMiningProject/CSI_Project')
    if not (BASE_DIR / 'config.yaml').exists():
        BASE_DIR = Path('/content/drive/MyDrive/CSI_Project')
    print(f'Colab -- BASE_DIR: {BASE_DIR}')
else:
    BASE_DIR = Path(os.path.dirname(os.path.abspath('__file__')))
    if not (BASE_DIR / 'datasets').exists():
        BASE_DIR = Path.cwd()
    print(f'Local -- BASE_DIR: {BASE_DIR}')

cfg_path = BASE_DIR / 'config.yaml'
assert cfg_path.exists(), f'Missing config.yaml at {cfg_path}'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

UNIFIED_JSONL   = BASE_DIR / cfg['unified_jsonl']
TOKEN_CACHE_DIR = BASE_DIR / cfg['token_cache_dir']
TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

ANAS_CACHE      = TOKEN_CACHE_DIR / 'tokens_maxlen512.pt'
VULBERTA_CACHE  = TOKEN_CACHE_DIR / 'tokens_vulberta_maxlen512.pt'
VULBERTA_NAME   = 'claudios/VulBERTa-MLP-D2A'
MAX_LEN         = 512
BATCH_SIZE      = 32

assert UNIFIED_JSONL.exists(), f'Missing UNIFIED.jsonl at {UNIFIED_JSONL}'
assert ANAS_CACHE.exists(),    f'Missing Anas cache at {ANAS_CACHE}'

print(f'ANAS_CACHE     : {ANAS_CACHE.name}  ({ANAS_CACHE.stat().st_size/1e6:.0f} MB)')
print(f'VULBERTA_CACHE : {VULBERTA_CACHE.name}  (output)')
print(f'VULBERTA_NAME  : {VULBERTA_NAME}')
print(f'MAX_LEN        : {MAX_LEN}')
print(f'BATCH_SIZE     : {BATCH_SIZE}')


## 2 — Load Anas Cache + UNIFIED.jsonl

We load Anas's cache to get the chunk structure (global_ids, chunk_index, chunk_stride).
We load UNIFIED.jsonl to get the actual source code of each function.
Then we reconstruct each chunk's text and tokenize with VulBERTa.

In [ ]:
# Load Anas cache -- for chunk structure only
print('Loading Anas cache...')
anas = torch.load(ANAS_CACHE, weights_only=True)
print(f'  num_records   : {anas["num_records"]}')
print(f'  num_functions : {anas["num_functions"]}')
print(f'  chunk_stride  : {anas["chunk_stride"]}')
print(f'  keys          : {list(anas.keys())}')

# Load UNIFIED.jsonl -- for source code text
print('\nLoading UNIFIED.jsonl...')
records = []
with open(UNIFIED_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'  Functions loaded: {len(records):,}')
assert len(records) == anas['num_functions'], (
    f'Mismatch: UNIFIED.jsonl has {len(records)} functions '
    f'but Anas cache has {anas["num_functions"]} functions.'
)
print(f'  Function count matches Anas cache: True')

# CWE label map
CWE_LABEL_MAP = {
    'CWE-077': 0, 'CWE-601': 1, 'CWE-022': 2,
    'CWE-094': 3, 'CWE-089': 4, 'CWE-352': 5, 'CWE-079': 6,
}


## 3 — Reconstruct Chunks + Tokenize with VulBERTa

For each of the 14,522 chunks in Anas's cache, we:
1. Look up the function text from UNIFIED.jsonl using `global_id`
2. Apply the same chunking (token window with `chunk_stride`) 
3. Tokenize that chunk's text with VulBERTa tokenizer

This guarantees record `i` in VulBERTa cache = same code window as record `i` in Anas's cache.

**Time estimate:** ~8-10 minutes on Colab T4 for 14,522 chunks.

In [ ]:
from tqdm import tqdm
from transformers import AutoTokenizer

if VULBERTA_CACHE.exists():
    print(f'Already exists: {VULBERTA_CACHE.name}  ({VULBERTA_CACHE.stat().st_size/1e6:.0f} MB)')
    print('Skipping. Delete file and re-run to rebuild.')
else:
    # Load VulBERTa tokenizer
    print(f'Loading VulBERTa tokenizer: {VULBERTA_NAME}')
    vulb_tok = AutoTokenizer.from_pretrained(VULBERTA_NAME)
    print(f'  Vocab size       : {vulb_tok.vocab_size:,}')
    print(f'  Model max length : {vulb_tok.model_max_length}')

    # Fix missing pad token if needed
    if vulb_tok.pad_token is None:
        vulb_tok.pad_token = vulb_tok.eos_token
        print(f'  pad_token set to eos: {vulb_tok.pad_token!r}')

    # Get chunk parameters from Anas's cache
    global_ids   = anas['global_ids']    # (14522,) -- which function
    chunk_indices= anas['chunk_index']   # (14522,) -- which chunk of that function
    chunk_counts = anas['chunk_count']   # (14522,) -- total chunks for that function
    chunk_stride = int(anas['chunk_stride'])  # overlap between chunks
    split_origins= anas['split_origins'] # (14522,) -- train/val

    print(f'\nProcessing {len(global_ids):,} chunks...')
    print(f'  chunk_stride = {chunk_stride} tokens')
    print(f'  This means each chunk overlaps by {chunk_stride} tokens with the next')

    all_ids, all_mask = [], []

    # Process in batches
    batch_texts = []

    for i in tqdm(range(len(global_ids)), desc='Building chunk texts'):
        gid     = global_ids[i].item()
        c_idx   = chunk_indices[i].item()
        c_count = chunk_counts[i].item()

        # Get full function text
        fn_text = ' '.join(records[gid]['lines'])

        if c_count == 1:
            # Single chunk -- use full text, tokenizer will truncate
            batch_texts.append(fn_text)
        else:
            # Multi-chunk function -- extract the right window
            # Tokenize full text first to get all tokens
            full_tokens = vulb_tok.encode(fn_text, add_special_tokens=False)
            # Window: chunk_idx * (MAX_LEN - 2 - chunk_stride) to that + (MAX_LEN - 2)
            # -2 accounts for [CLS] and [SEP] special tokens
            window_size = MAX_LEN - 2
            step        = window_size - chunk_stride
            start       = c_idx * step
            end         = start + window_size
            chunk_token_ids = full_tokens[start:end]
            # Decode back to text for the tokenizer call
            chunk_text = vulb_tok.decode(chunk_token_ids, skip_special_tokens=True)
            batch_texts.append(chunk_text)

        # Tokenize when batch is full or at the end
        if len(batch_texts) == BATCH_SIZE or i == len(global_ids) - 1:
            enc = vulb_tok(
                batch_texts,
                max_length            = MAX_LEN,
                padding               = 'max_length',
                truncation            = True,
                return_tensors        = 'pt',
                return_attention_mask = True,
            )
            all_ids.append(enc['input_ids'])
            all_mask.append(enc['attention_mask'])
            batch_texts = []

    # Stack
    input_ids      = torch.cat(all_ids,  dim=0)  # (14522, 512)
    attention_mask = torch.cat(all_mask, dim=0)  # (14522, 512)
    print(f'\nTokenization complete:')
    print(f'  input_ids shape : {tuple(input_ids.shape)}')
    print(f'  attn_mask shape : {tuple(attention_mask.shape)}')

    # Build label tensors -- one per chunk (same label as the function)
    cwe_labels = torch.tensor(
        [CWE_LABEL_MAP.get(records[gid.item()].get('cwe_id', ''), -1)
         for gid in global_ids],
        dtype=torch.long
    )
    binary_labels = torch.tensor(
        [int(records[gid.item()]['is_vulnerable']) for gid in global_ids],
        dtype=torch.long
    )

    # Save
    cache = {
        'input_ids':      input_ids,
        'attention_mask': attention_mask,
        'cwe_labels':     cwe_labels,
        'binary_labels':  binary_labels,
        'global_ids':     global_ids,
        'split_origins':  split_origins,
        'chunk_index':    chunk_indices,
        'chunk_count':    chunk_counts,
        'num_records':    len(global_ids),
        'num_functions':  anas['num_functions'],
        'model_name':     VULBERTA_NAME,
        'max_length':     MAX_LEN,
    }
    torch.save(cache, VULBERTA_CACHE)

    size_mb  = VULBERTA_CACHE.stat().st_size / 1e6
    avg_real = attention_mask.float().sum(dim=1).mean().item()
    print(f'\nSaved: {VULBERTA_CACHE.name}')
    print(f'  Size            : {size_mb:.0f} MB')
    print(f'  Records         : {len(global_ids):,}')
    print(f'  Avg real tokens : {avg_real:.1f} / {MAX_LEN}')


## 4 — Verification

Confirm VulBERTa cache matches Anas's GCB cache exactly:
same number of records, same global_ids, same split_origins, different token IDs.

In [ ]:
print('=' * 55)
print('VERIFICATION')
print('=' * 55)

vulb_c = torch.load(VULBERTA_CACHE, weights_only=True)
N      = anas['num_records']

# [1] Shapes
print('\n[1] Shapes...')
assert vulb_c['input_ids'].shape == (N, MAX_LEN), \
    f'Shape wrong: {vulb_c["input_ids"].shape} != ({N}, {MAX_LEN})'
assert anas['input_ids'].shape   == (N, MAX_LEN)
print(f'  Anas GCB    : {tuple(anas["input_ids"].shape)}')
print(f'  VulBERTa    : {tuple(vulb_c["input_ids"].shape)}')
print(f'  PASS')

# [2] Different vocabularies
print('\n[2] Vocabularies are different...')
ids_differ = (anas['input_ids'] != vulb_c['input_ids']).any().item()
assert ids_differ, 'ERROR: identical token IDs -- same tokenizer used!'
pct = (anas['input_ids'] != vulb_c['input_ids']).float().mean().item() * 100
print(f'  {pct:.1f}% of token positions differ -- correct')
print(f'  PASS')

# [3] Alignment
print('\n[3] Alignment...')
assert vulb_c['num_records'] == anas['num_records']
assert vulb_c['split_origins'] == anas['split_origins'], \
    'split_origins mismatch'
assert torch.equal(
    vulb_c['global_ids'],
    anas['global_ids']
), 'global_ids mismatch'
assert torch.equal(
    vulb_c['chunk_index'],
    anas['chunk_index']
), 'chunk_index mismatch'
print(f'  num_records    : {vulb_c["num_records"]:,}  PASS')
print(f'  split_origins  : aligned  PASS')
print(f'  global_ids     : aligned  PASS')
print(f'  chunk_index    : aligned  PASS')

# [4] No empty sequences
print('\n[4] No empty sequences...')
empty = (vulb_c['attention_mask'].sum(dim=1) == 0).sum().item()
assert empty == 0, f'{empty} VulBERTa sequences are all-padding'
avg = vulb_c['attention_mask'].float().sum(dim=1).mean().item()
print(f'  Avg real tokens/chunk : {avg:.1f} / {MAX_LEN}')
print(f'  PASS')

print('\n' + '=' * 55)
print('ALL CHECKS PASSED')
print('=' * 55)
print()
print('Summary:')
print(f'  Anas GCB cache   : {ANAS_CACHE.name}  ({ANAS_CACHE.stat().st_size/1e6:.0f} MB)')
print(f'  VulBERTa cache   : {VULBERTA_CACHE.name}  ({VULBERTA_CACHE.stat().st_size/1e6:.0f} MB)')
print(f'  Records in both  : {vulb_c["num_records"]:,}  (14,522 chunks of 4,085 functions)')
print()
print('Next steps:')
print('  1. Update VulBERTaFusionModel to use claudios/VulBERTa-MLP-D2A')
print('  2. Update DualCacheDataset to load these two caches')
print('  3. Run 06_dual_encoder_training.ipynb')
